# Presenter Demo: три площадки за 7—10 минут

> **Курс «От нуля до своих агентов» — Модуль 1.5.**
>
> Это демонстрационный ноутбук для урока. После **каждой** ячейки
> кода идёт абзац-разбор: что код сделал, что значит вывод, и что
> можно **поменять самому**, чтобы попробовать.
>
> Цель: к концу ноутбука вы понимаете, чем три площадки (Kaggle,
> Hugging Face, Google AI Studio) отличаются друг от друга — на
> своих руках, не на словах.
>
> **Перед запуском:** в Kaggle → `Add-ons → Secrets` должен быть
> прикреплён `GOOGLE_API_KEY`. `HF_TOKEN` не обязателен.

# Часть 1. Kaggle — где живёт код

Kaggle — это **бесплатный компьютер в браузере**. Вам не надо ставить
Python, не надо настраивать видеокарту, не надо платить за облако.
Вы открыли вкладку — у вас уже есть рабочая среда с Python, GPU
(если попросить) и местом на диске.

Давайте посмотрим, что именно нам дали.

In [ ]:
import sys, platform, os

print("Версия Python:", sys.version.split()[0])
print("Операционка:  ", platform.platform())
print("Где мы сейчас:", os.getcwd())

# Сколько места осталось в папке, которая переживёт перезапуск ноутбука
free_gb = os.statvfs('/kaggle/working').f_bavail * 512 // (1024**3)
print(f"Свободно в /kaggle/working: {free_gb} ГБ")

**Что мы сейчас увидели:**

- **Python 3.11+** — современная версия, всё свежее.
- **Linux** — Kaggle крутится на Linux-серверах. Если у вас Windows
  или Mac — на Kaggle всё одно ведёт себя одинаково.
- **`/kaggle/working`** — это ваша личная папка. Всё, что вы сюда
  сохраните, **переживёт перезапуск ноутбука** (а вот переменные в
  памяти — нет).
- **20 ГБ места** — этого хватит на любую учебную модель.

**Поменять и попробовать:**

- Добавьте свою строку, например `print("Привет!")`.
- Или замените `/kaggle/working` на `/tmp` — увидите, сколько места
  во временной папке (она *не* переживает перезапуск, но больше по
  объёму).

In [ ]:
# Проверяем, дали ли нам видеокарту
!nvidia-smi -L 2>/dev/null || echo "GPU не подключён — это нормально, переключите Accelerator в правой панели если нужен"

**Что произошло:**

`!nvidia-smi -L` — это **команда операционной системы** (восклицательный
знак `!` в Jupyter означает «запусти не Python, а shell»). Она спрашивает
видеокарту: «представься».

- Если видишь строку вида `GPU 0: Tesla T4 (UUID: ...)` — отлично,
  у тебя есть видеокарта.
- Если видишь «GPU не подключён» — справа в `Notebook options`
  переключи **Accelerator** на `GPU T4 x2` или оставь `None`. Для
  сегодняшнего демо GPU **не обязателен** — модели крошечные, на
  CPU летают.

**Поменять и попробовать:**

- Запусти `!ls /kaggle/working` — увидишь, что в твоей папке лежит.
- Запусти `!pip list | head -20` — увидишь первые 20 установленных
  библиотек (их там несколько сотен по умолчанию).

# Часть 2. Hugging Face — где живут модели

**Hugging Face (HF)** — это сайт, где люди делятся моделями
машинного обучения. Похоже на GitHub, но вместо кода — обученные
модели (готовые «мозги», которым уже не надо учиться, можно сразу
использовать).

Мы возьмём оттуда две модели:

1. **`distilbert`** — определяет настроение текста (positive /
   negative). Маленькая, ~250 МБ.
2. **`MiniLM`** — превращает текст в «вектор» (набор чисел), по
   которому можно сравнивать тексты между собой по смыслу. Ещё
   меньше, ~90 МБ.

In [ ]:
# Ставим библиотеку transformers (это инструмент для работы с моделями HF)
!pip install -q transformers

**Что произошло:** `!pip install -q transformers` — установили
библиотеку. `-q` означает «тихо» (без длинного лога установки).

Эта команда обычно занимает 10—20 секунд. Запускается **один раз** на
ноутбук — после установки библиотека уже в памяти Kaggle.

In [ ]:
from transformers import pipeline

# device=-1 означает "считай на CPU". У нас модели маленькие, GPU не нужен.
clf = pipeline("sentiment-analysis", device=-1)

# Кормим модели три фразы и смотрим, что она думает
print(clf("This course is unexpectedly fun."))
print(clf("Honestly, I'm just here for the certificate."))
print(clf("It was not bad, actually."))

**Что произошло — построчно:**

1. `from transformers import pipeline` — импортировали готовый
   «конвейер». Этот объект сам качает модель с HF, готовит её к работе
   и принимает текст на вход.
2. `pipeline("sentiment-analysis", device=-1)` — попросили конвейер
   для задачи «определить настроение». HF сам выбрал лучшую
   дефолтную модель — это `distilbert`. `device=-1` = CPU.
3. `clf("This course is unexpectedly fun.")` — отдали фразу модели,
   она вернула ответ.

**Что значит вывод:**

```
[{'label': 'POSITIVE', 'score': 0.9998}]
[{'label': 'POSITIVE', 'score': 0.7815}]
[{'label': 'POSITIVE', 'score': 0.9993}]
```

- **`label`** — что модель решила: POSITIVE или NEGATIVE.
- **`score`** — насколько модель уверена (от 0 до 1, где 1 — «точно»).

**Интересный момент:** вторая фраза «I'm just here for the certificate»
получила score **0.78** — модель сомневается. На самом деле тут лёгкий
сарказм («я тут только ради сертификата»), но distilbert этого не
улавливает и говорит «вроде позитивная». Это пример **границ модели**:
маленькая специализированная модель отлично справляется с очевидным,
но путается на тонкостях.

**Поменять и попробовать:**

- Добавь ещё одну фразу: `print(clf("This is terrible!"))` —
  должно вернуть NEGATIVE с высоким score.
- Попробуй на русском: `print(clf("Какой ужасный урок"))` — увидишь,
  что модель растеряется (она училась на английском).
- Длинный отзыв: `print(clf("I loved the beginning but the ending was awful."))` —
  посмотри, что модель решит при смешанной оценке.

In [ ]:
# Вторая модель — превращает текст в "вектор" (точку в многомерном пространстве)
from sentence_transformers import SentenceTransformer
import numpy as np

emb_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)

phrases = [
    "The quick brown fox jumps over the lazy dog.",
    "A fast fox leaps over a sleepy hound.",         # то же самое, другими словами
    "I love pizza with pineapple.",                   # совсем про другое
    "Шустрый лис прыгает через ленивую собаку.",     # русский перевод первой фразы
]

# Превращаем каждую фразу в вектор из 384 чисел
vectors = emb_model.encode(phrases, normalize_embeddings=True)

# Считаем "похожесть" каждой пары: близко к 1.0 = очень похожи, к 0 = разные
similarity = vectors @ vectors.T

# Красивая таблица
import pandas as pd
labels = [p[:40] + ("..." if len(p) > 40 else "") for p in phrases]
pd.DataFrame(similarity.round(2), index=labels, columns=labels)

**Что произошло — построчно:**

1. **Скачали** ещё одну модель — `all-MiniLM-L6-v2`. Это
   embedding-модель: на вход — текст, на выход — список из 384 чисел
   («вектор» или «эмбеддинг»). Близкие по смыслу фразы дают близкие
   векторы.
2. **Дали ей 4 фразы**: оригинал про лиса, перефраз про лиса, фраза
   про пиццу (не в тему), и тот же лис на русском.
3. **`emb_model.encode(...)`** — превратили каждую фразу в вектор.
4. **`vectors @ vectors.T`** — посчитали «cosine similarity» между
   каждой парой. Это математический способ сказать «насколько две
   точки рядом» (1.0 = совпадение, 0 = совсем не похоже).

**Что значит таблица:**

```
                              fox|fast fox|pizza|русский лис
the quick brown fox        | 1.00 | 0.76  | 0.04 | 0.10
a fast fox leaps           | 0.76 | 1.00  | 0.02 | 0.13
i love pizza               | 0.04 | 0.02  | 1.00 | 0.00
Шустрый лис прыгает        | 0.10 | 0.13  | 0.00 | 1.00
```

- На **диагонали — 1.00** (каждая фраза идентична самой себе, очевидно).
- **0.76** между «the quick brown fox» и «a fast fox leaps» — модель
  поняла, что это **синонимы**. Хотя слова разные, смысл один.
- **0.04** между «лис» и «пицца» — модель видит, что фразы про
  совсем разное.
- **0.10** между английским «the quick brown fox» и русским
  переводом — а вот тут модель **сломалась**. Она училась только
  на английском, поэтому русский для неё — просто «непонятный набор
  букв», и она не видит связи.

**Это важный урок:** модель ровно настолько умна, насколько ей
показали разнообразные данные при обучении. Если хотите, чтобы
работала и на русском — нужна **многоязычная** embedding-модель,
например `paraphrase-multilingual-MiniLM-L12-v2`.

**Поменять и попробовать:**

- Замени одну из фраз на свою — например `"My cat is sleeping on the keyboard."` —
  посмотри, на что она похожа больше всего.
- Добавь пятую фразу-перефраз: `"A cat is napping on a laptop."` —
  должно дать ~0.7 с предыдущей.
- Поменяй модель на многоязычную:
  `SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu")` —
  и русская фраза станет похожа на английскую.

# Часть 3. Google AI Studio — где живёт Gemini

**Google AI Studio** — это сайт Google, на котором живёт большая
языковая модель **Gemini**. В отличие от Hugging Face, Gemini не
скачивается к вам на компьютер — это **облачная модель**.

Вы шлёте ей текст-вопрос (или картинку), она шлёт ответ. Платите
обычно за каждый запрос (но небольшой объём в день — бесплатно).

Главное отличие от distilbert и MiniLM: Gemini — **универсальная**.
Она может всё подряд (классифицировать, переводить, писать код,
читать картинки), а не одну задачу.

In [ ]:
# Установка библиотеки для общения с Gemini
!pip install -q google-genai

**Что произошло:** установили библиотеку `google-genai` —
официальный инструмент Google для отправки запросов к Gemini из Python.

In [ ]:
import enum
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient

# Достаём API-ключ из Kaggle Secrets (не из кода!)
client = genai.Client(api_key=UserSecretsClient().get_secret("GOOGLE_API_KEY"))


# Описываем словами Gemini, какие ответы мы допускаем.
# enum.Enum в Python — это "выбор из списка вариантов".
class Sentiment(enum.Enum):
    POSITIVE = "POSITIVE"
    NEUTRAL = "NEUTRAL"
    NEGATIVE = "NEGATIVE"


# Настройки: модель должна вернуть только одно из трёх значений выше,
# никакого свободного текста.
cfg = types.GenerateContentConfig(
    response_mime_type="text/x.enum",
    response_schema=Sentiment,
    temperature=0,  # 0 = без креатива, всегда одинаковый ответ
)

# Отправляем 4 фразы и печатаем результат
for phrase in [
    "It was not bad, actually.",
    "Лучше бы я остался дома.",
    "норм",
    "Best class I've taken in years!",
]:
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        config=cfg,
        contents=phrase,
    )
    print(f"{resp.parsed.value:8s} | {phrase}")

**Что произошло — построчно:**

1. **`UserSecretsClient().get_secret("GOOGLE_API_KEY")`** — достали
   API-ключ из Kaggle Secrets. Это **безопасный способ**: ключ не
   виден в коде, его не увидят те, кому вы покажете ноутбук.
2. **`class Sentiment(enum.Enum)`** — сказали Python: «возможные
   ответы — только эти три слова». Это нужно для следующего шага.
3. **`response_schema=Sentiment`** — сказали Gemini: «отвечай только
   одним словом из этого списка, никаких объяснений, никаких
   `I think it's POSITIVE because...`». Это называется **structured
   output** и сильно облегчает работу: вы заранее знаете, какой ответ
   получите.
4. **`temperature=0`** — отключили креатив. Модель всегда даёт тот
   же ответ на тот же вопрос. (При `temperature=1` модель чуть-чуть
   импровизирует, при `2` — фантазирует сильно.)
5. **Цикл по 4 фразам** — каждую отправляем в Gemini, получаем
   ответ, печатаем.

**Что значит вывод:**

```
POSITIVE | It was not bad, actually.
NEGATIVE | Лучше бы я остался дома.
POSITIVE | норм
POSITIVE | Best class I've taken in years!
```

- **«It was not bad, actually.»** — фраза с двойным отрицанием,
  по смыслу позитивная. Gemini её **понял правильно**, в отличие от
  distilbert, который часто на таком ошибается.
- **«Лучше бы я остался дома.»** — на русском, чистый негатив.
  Gemini понимает русский — это плюс ко всему.
- **«норм»** — одно слово на русском сленге. Gemini понял, что это
  скорее позитив. distilbert тут вообще ничего бы не сказал
  осмысленно.
- **Последняя** — очевидный позитив, никаких сюрпризов.

**Вывод:** Gemini сильно лучше понимает язык, особенно тонкости и
русский. Платите за это: каждый запрос идёт в облако Google, медленнее
distilbert и стоит денег при большом объёме.

**Поменять и попробовать:**

- Добавь свою фразу с сарказмом: «I love waiting in line for hours.»
- Поменяй `temperature=0` на `temperature=2` — увидишь, что модель
  иногда «передумывает».
- Добавь в `Sentiment` четвёртый вариант: `SARCASTIC = "SARCASTIC"` —
  Gemini научится распознавать сарказм отдельной категорией.

## Бонус: Gemini «видит» картинку

Gemini умеет не только читать текст, но и **смотреть на картинки**.
Покажем ему фото чека и попросим вытащить оттуда название магазина,
сумму и количество товаров — **сразу в виде JSON**, готовый для
программы.

In [ ]:
import typing_extensions as typing
import urllib.request

# Скачиваем публичный пример чека (картинка лежит на HF)
url = "https://datasets-server.huggingface.co/assets/mychen76/invoices-and-receipts_ocr_v1/--/default/test/0/image/image.jpg"
img_bytes = urllib.request.urlopen(url).read()


# Описываем, какой JSON мы хотим получить
class Receipt(typing.TypedDict):
    vendor: str          # название магазина
    total_amount: str    # итоговая сумма (строкой, потому что у чеков валюта в формате)
    item_count: int      # сколько позиций


# Отправляем картинку + просьбу извлечь поля
resp = client.models.generate_content(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Receipt,
        temperature=0,
    ),
    contents=[
        types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"),
        "Extract vendor name, total amount, and number of line items from this receipt.",
    ],
)
print(resp.text)

**Что произошло:**

1. **Скачали картинку** чека с публичного датасета HF.
2. **Описали JSON-схему** ответа: три поля — название магазина,
   сумма, число товаров.
3. **Отправили** Gemini две вещи: саму картинку + просьбу «вытащи
   эти три поля».
4. Gemini **прочитал чек глазами**, нашёл нужные поля, вернул
   готовый JSON.

**Что значит вывод:**

Результат будет выглядеть примерно так:
```json
{"vendor": "Costco Wholesale", "total_amount": "$45.67", "item_count": 8}
```

(точные значения зависят от того, какой чек попался — он рандомный
из публичного датасета).

**Почему это круто:** ещё 2 года назад «прочитать чек и достать поля»
требовало OCR-библиотеку (Tesseract), регулярные выражения и неделю
программирования. Сейчас — **один запрос к LLM**. Это и есть та
самая «революция foundation-моделей», о которой мы говорили в
Модуле 2.

**Поменять и попробовать:**

- Добавь в `Receipt` поле `date: str` — Gemini вытащит и дату.
- Поменяй просьбу на «Translate the vendor name to Russian» — модель
  ещё и переведёт.
- Дай свою картинку (загрузи через `+ Add Input → Upload`) и
  попроси извлечь из неё что-нибудь.

# Часть 4. Бонус-бонус: function calling — первый агент

Это уже **тизер к Модулю 8**. Покажем, как LLM может **сама вызывать
функции**, которые мы написали, чтобы достать данные.

In [ ]:
import sqlite3

# Создаём крошечную базу в памяти — представим, что это каталог магазина
db = sqlite3.connect(":memory:")
db.executescript('''
CREATE TABLE products (name TEXT, price REAL);
INSERT INTO products VALUES ('Laptop', 799.99), ('Keyboard', 129.99), ('Mouse', 29.99);
''')


def list_products() -> list[tuple[str, float]]:
    """Return all products with their prices from the store catalog."""
    print(" -> Gemini позвал нашу функцию list_products()")
    return db.execute("SELECT name, price FROM products").fetchall()


# Создаём чат, который умеет звать нашу функцию когда понадобится
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are a sales assistant. Use the tool to look up products.",
        tools=[list_products],
    ),
)

# Задаём вопрос на русском — модель сама поймёт, что надо звать функцию
print(chat.send_message("Какой у нас самый дешёвый товар и сколько он стоит?").text)

**Что произошло — это самое важное в демо:**

1. **Создали базу данных** в памяти — 3 товара с ценами.
2. **Написали обычную Python-функцию** `list_products()` — она
   возвращает все товары. Главное здесь — **docstring** в тройных
   кавычках сразу после `def`. Gemini читает этот docstring и решает,
   когда функцию надо вызвать.
3. **Создали чат** и передали ему функцию через `tools=[list_products]`.
4. **Задали вопрос на русском.** Gemini:
   - Прочитал вопрос.
   - Подумал: «надо посмотреть товары».
   - **Сам решил позвать** `list_products()`.
   - Получил данные `[('Laptop', 799.99), ('Keyboard', 129.99), ('Mouse', 29.99)]`.
   - Выбрал самый дешёвый (Mouse).
   - Ответил на русском человеческим языком.

**Это и есть агент.** Не «LLM, которая отвечает текстом», а
«LLM, которая может звать функции и взаимодействовать с миром».

В Модуле 8 мы это разберём подробно: что такое цикл, что такое
инструменты, как сделать так, чтобы агент не наделал ерунды.

**Поменять и попробовать:**

- Спроси по-другому: «What's our most expensive product?»
- Добавь второй tool — функцию `add_product(name, price)` и попроси
  «добавь чай за 5 долларов».
- Меняй вопрос: «У нас есть клавиатуры?», «Сколько товаров в магазине?»

# Что мы только что увидели

За эти 7—10 минут мы:

1. **Kaggle** — нашли свою рабочую среду, проверили Python и место
   на диске.
2. **Hugging Face** — скачали две готовые модели, прогнали через них
   тексты, увидели и сильные стороны (быстро, дёшево), и слабые
   (не понимают русский, не ловят сарказм).
3. **Google AI Studio** — поговорили с Gemini, заставили его отвечать
   только enum'ами, прочитали картинку чека в JSON.
4. **Function calling** — увидели, как Gemini сам вызывает наш Python-код.

Это все три площадки в работе. Дальше на курсе будем нырять в каждую
глубже — но базу вы только что поглядели **руками**.

**Что попробовать сразу:**

- Любая ячейка выше — поменяйте текст / фразу / промпт, нажмите
  `Shift+Enter`. Это никак ничего не сломает (хуже всего —
  получите смешной ответ).
- `Run All` ещё раз — все скачивания уже в кэше, второй прогон будет
  в несколько раз быстрее.